# Testing with pytest
### Test functions · Assertions · Arrange–Act–Assert · Normal, edge & invalid cases

**What you will learn**

1. Why we write automated tests
2. How to write and run **test functions** with pytest
3. How pytest **discovers** tests (naming rules)
4. Writing powerful **assertions**: equality, collections, floats, exceptions, custom messages
5. Structuring every test with the **Arrange–Act–Assert (AAA)** pattern
6. Choosing test inputs: **normal cases, edge cases and invalid input**
7. Useful command-line options for running tests
8. Hands-on exercises (with solutions at the end)

> **How this notebook works:** pytest runs test *files*, so we use the `%%writefile` magic to create small `.py` files, then run pytest with `!python -m pytest ...`. Run the cells from top to bottom.

## 1. Setup

Install pytest (only needed once) and check the version.

In [1]:
!pip install -q pytest
!python -m pytest --version

pytest 9.1.1


In [ ]:
# Create a clean working folder for this lesson so our files don't mix with others
import os
os.makedirs("pytest_lesson", exist_ok=True)
os.chdir("pytest_lesson")
print("Working in:", os.getcwd())

## 2. Why test?

A **test** is code that checks that other code behaves as expected.

| Without tests | With tests |
|---|---|
| You click around / print values to check things | One command checks everything in seconds |
| A change in one place silently breaks another | Broken behaviour is caught immediately |
| Fear of refactoring | Confidence to improve code |
| Behaviour lives only in your head | Tests document how the code should behave |

**pytest** is the most popular Python testing framework because tests are just plain functions with plain `assert` statements: no classes or special methods required.

## 3. Your first test function

### 3.1 The code we want to test

In [3]:
%%writefile calculator.py
def add(a, b):
    return a + b


def subtract(a, b):
    return a - b


def multiply(a, b):
    return a * b


def divide(a, b):
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

Writing calculator.py

### 3.2 The tests

A pytest **test function** is simply:

- a function whose name **starts with `test_`**
- inside a file whose name **starts with `test_`** (or ends with `_test.py`)
- that uses **`assert`** to check a condition

If every `assert` is true, the test **passes**. If any is false (or an unexpected exception is raised), it **fails**.

In [4]:
%%writefile test_calculator.py
from calculator import add, subtract, multiply, divide


def test_add_two_positive_numbers():
    assert add(2, 3) == 5


def test_add_negative_numbers():
    assert add(-2, -3) == -5


def test_subtract():
    assert subtract(10, 4) == 6


def test_multiply_by_zero():
    assert multiply(7, 0) == 0


def test_divide():
    assert divide(10, 2) == 5

Writing test_calculator.py


### 3.3 Run the tests

`-v` (verbose) prints one line per test.

In [5]:
!python -m pytest test_calculator.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-9.1.1, pluggy-1.6.0 -- /usr/local/bin/python
cachedir: .pytest_cache
rootdir: /tmp/claude-0/-home-claude/46131263-cd34-5556-b5d9-de7c2ec56ddb/scratchpad/nb/pytest_lesson
plugins: platformdirs-4.13.5, anyio-4.15.1
collected 5 items                                                              

test_calculator.py::test_add_two_positive_numbers PASSED                 [ 20%]
test_calculator.py::test_add_negative_numbers PASSED                     [ 40%]
test_calculator.py::test_subtract PASSED                                 [ 60%]
test_calculator.py::test_multiply_by_zero PASSED                         [ 80%]
test_calculator.py::test_divide PASSED                                   [100%]

============================== 5 passed in 0.01s ===============================


**Reading the output**

- `PASSED` / `FAILED` per test, with its **node id**: `file::test_name`
- A dot `.` per passing test when not in verbose mode, `F` for a failure, `E` for an error
- A summary line at the bottom, e.g. `5 passed in 0.01s`

> 💡 Good test names describe **behaviour**: `test_add_negative_numbers` tells you exactly what broke when it fails. Avoid names like `test1` or `test_add_works`.

## 4. How pytest discovers tests

When you run `pytest` with no arguments, it searches the current folder (and subfolders) using these rules:

| What | Rule | Example |
|---|---|---|
| Test files | `test_*.py` or `*_test.py` | `test_calculator.py` |
| Test functions | name starts with `test` | `def test_add():` |
| Test classes | name starts with `Test`, **no** `__init__` | `class TestCart:` |
| Test methods | name starts with `test` inside a `Test` class | `def test_total(self):` |

Functions that don't follow the rules are ignored, which is handy for helper functions.

In [6]:
%%writefile test_discovery.py
def helper_make_list():          # not collected: doesn't start with "test"
    return [1, 2, 3]


def test_list_length():          # collected
    assert len(helper_make_list()) == 3


class TestStrings:               # collected: class starts with "Test"
    def test_upper(self):
        assert "hello".upper() == "HELLO"

    def test_strip(self):
        assert "  hi  ".strip() == "hi"

Writing test_discovery.py


In [7]:
# --collect-only lists what pytest WOULD run, without running it
!python -m pytest test_discovery.py --collect-only -q

test_discovery.py::test_list_length
test_discovery.py::TestStrings::test_upper
test_discovery.py::TestStrings::test_strip

3 tests collected in 0.00s


## 5. Assertions in depth

pytest uses Python's ordinary `assert` statement. Its superpower is **assertion rewriting**: when an assert fails, pytest shows you the actual values involved, not just `AssertionError`.

### 5.1 What a failure looks like

Let's deliberately write a failing test to learn how to read the report.

In [8]:
%%writefile test_failure_demo.py
def apply_discount(price, percent):
    return price - price * percent / 10      # BUG: should be / 100


def test_ten_percent_discount():
    assert apply_discount(200, 10) == 180

Writing test_failure_demo.py


In [9]:
!python -m pytest test_failure_demo.py

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-9.1.1, pluggy-1.6.0
rootdir: /tmp/claude-0/-home-claude/46131263-cd34-5556-b5d9-de7c2ec56ddb/scratchpad/nb/pytest_lesson
plugins: platformdirs-4.13.5, anyio-4.15.1
collected 1 item                                                               

test_failure_demo.py F                                                   [100%]

=================================== FAILURES ===================================
__________________________ test_ten_percent_discount ___________________________

    def test_ten_percent_discount():
>       assert apply_discount(200, 10) == 180
E       assert 0.0 == 180
E        +  where 0.0 = apply_discount(200, 10)

test_failure_demo.py:6: AssertionError
=========================== short test summary info ============================
FAILED test_failure_demo.py::test_ten_percent_discount - assert 0.0 == 180
============================== 1 fai

**How to read it:**

- The `>` marker points to the failing line.
- `E   assert 0.0 == 180` shows the **actual** value on the left and **expected** on the right.
- `where 0.0 = apply_discount(200, 10)` explains where the value came from.

Fix the bug (`/ 100`) and the test passes. This is the core loop of testing: **red → fix → green**.

### 5.2 Common assertion patterns

In [10]:
%%writefile test_assertions.py
def test_equality():
    assert 2 + 2 == 4
    assert "py" + "test" == "pytest"


def test_inequality_and_comparison():
    assert 5 != 3
    assert 10 > 3
    assert 3 <= 3


def test_truthiness():
    assert [1, 2]          # non-empty list is truthy
    assert not []          # empty list is falsy
    assert None is None


def test_membership():
    fruits = ["apple", "banana", "mango"]
    assert "mango" in fruits
    assert "grape" not in fruits
    assert "ell" in "hello"


def test_types():
    assert isinstance(3.14, float)
    assert isinstance({"a": 1}, dict)


def test_collections():
    assert [1, 2, 3] == [1, 2, 3]                  # order matters for lists
    assert {1, 2, 3} == {3, 2, 1}                  # not for sets
    assert {"name": "Ali", "age": 30} == {"age": 30, "name": "Ali"}
    assert len("pytest") == 6

Writing test_assertions.py


In [11]:
!python -m pytest test_assertions.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-9.1.1, pluggy-1.6.0 -- /usr/local/bin/python
cachedir: .pytest_cache
rootdir: /tmp/claude-0/-home-claude/46131263-cd34-5556-b5d9-de7c2ec56ddb/scratchpad/nb/pytest_lesson
plugins: platformdirs-4.13.5, anyio-4.15.1
collected 6 items                                                              

test_assertions.py::test_equality PASSED                                 [ 16%]
test_assertions.py::test_inequality_and_comparison PASSED                [ 33%]
test_assertions.py::test_truthiness PASSED                               [ 50%]
test_assertions.py::test_membership PASSED                               [ 66%]
test_assertions.py::test_types PASSED                                    [ 83%]
test_assertions.py::test_collections PASSED                              [100%]

============================== 6 passed in 0.02s ===============================


### 5.3 Rich diffs for collections

When lists, dicts or long strings differ, pytest pinpoints **exactly** what is different. Run this failing example and look at the diff.

In [12]:
%%writefile test_diff_demo.py
def get_user():
    return {"name": "Ali", "role": "admin", "active": True, "city": "Lahore"}


def test_user_profile():
    expected = {"name": "Ali", "role": "editor", "active": True, "city": "Lahore"}
    assert get_user() == expected

Writing test_diff_demo.py


In [13]:
!python -m pytest test_diff_demo.py -vv

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-9.1.1, pluggy-1.6.0 -- /usr/local/bin/python
cachedir: .pytest_cache
rootdir: /tmp/claude-0/-home-claude/46131263-cd34-5556-b5d9-de7c2ec56ddb/scratchpad/nb/pytest_lesson
plugins: platformdirs-4.13.5, anyio-4.15.1
collected 1 item                                                               

test_diff_demo.py::test_user_profile 

FAILED                              [100%]

=================================== FAILURES ===================================
______________________________ test_user_profile _______________________________

    def test_user_profile():
        expected = {"name": "Ali", "role": "editor", "active": True, "city": "Lahore"}
>       assert get_user() == expected
E       AssertionError: assert {'name': 'Ali', 'role': 'admin', 'active': True, 'city': 'Lahore'} == {'name': 'Ali', 'role': 'editor', 'active': True, 'city': 'Lahore'}
E         
E         Common items:
E         {'active': True, 'city': 'Lahore', 'name': 'Ali'}
E         Differing items:
E         {'role': 'admin'} != {'role': 'editor'}
E         
E         Full diff:
E           {
E               'name': 'Ali',
E         -     'role': 'editor',
E         ?              ^  ^^^
E         +     'role': 'admin',
E         ?              ^ + ^
E               'active': True,
E               'city': 'Lahore',
E           }

test_diff_

Notice `Differing items: {'role': 'admin'} != {'role': 'editor'}`: pytest found the one wrong key for you. `-vv` (extra verbose) shows the full diff.

### 5.4 Comparing floats with `pytest.approx`

Floating-point arithmetic is not exact, so `==` can fail unexpectedly:

In [14]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)

0.30000000000000004
False


In [15]:
%%writefile test_floats.py
import pytest


def test_float_sum():
    assert 0.1 + 0.2 == pytest.approx(0.3)


def test_float_with_tolerance():
    assert 9.81 == pytest.approx(10, abs=0.2)       # within ±0.2
    assert 102 == pytest.approx(100, rel=0.05)      # within 5%


def test_approx_works_on_lists():
    assert [0.1 + 0.2, 0.2 + 0.4] == pytest.approx([0.3, 0.6])

Writing test_floats.py


In [16]:
!python -m pytest test_floats.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-9.1.1, pluggy-1.6.0 -- /usr/local/bin/python
cachedir: .pytest_cache
rootdir: /tmp/claude-0/-home-claude/46131263-cd34-5556-b5d9-de7c2ec56ddb/scratchpad/nb/pytest_lesson
plugins: platformdirs-4.13.5, anyio-4.15.1
collected 3 items                                                              

test_floats.py::test_float_sum PASSED                                    [ 33%]
test_floats.py::test_float_with_tolerance PASSED                         [ 66%]
test_floats.py::test_approx_works_on_lists PASSED                        [100%]

============================== 3 passed in 0.01s ===============================


### 5.5 Testing that an exception is raised: `pytest.raises`

Sometimes the **correct** behaviour is an error. Use `pytest.raises` as a context manager: the test passes only if the expected exception is raised inside the `with` block.

In [17]:
%%writefile test_exceptions.py
import pytest
from calculator import divide


def test_divide_by_zero_raises_value_error():
    with pytest.raises(ValueError):
        divide(10, 0)


def test_divide_by_zero_message():
    # match= checks the error message (it is a regular expression)
    with pytest.raises(ValueError, match="divide by zero"):
        divide(10, 0)


def test_inspect_the_exception():
    with pytest.raises(ValueError) as exc_info:
        divide(1, 0)
    assert str(exc_info.value) == "Cannot divide by zero"


def test_int_conversion_fails():
    with pytest.raises(ValueError):
        int("abc")

Writing test_exceptions.py


In [18]:
!python -m pytest test_exceptions.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-9.1.1, pluggy-1.6.0 -- /usr/local/bin/python
cachedir: .pytest_cache
rootdir: /tmp/claude-0/-home-claude/46131263-cd34-5556-b5d9-de7c2ec56ddb/scratchpad/nb/pytest_lesson
plugins: platformdirs-4.13.5, anyio-4.15.1
collected 4 items                                                              

test_exceptions.py::test_divide_by_zero_raises_value_error PASSED        [ 25%]
test_exceptions.py::test_divide_by_zero_message PASSED                   [ 50%]
test_exceptions.py::test_inspect_the_exception PASSED                    [ 75%]
test_exceptions.py::test_int_conversion_fails PASSED                     [100%]

============================== 4 passed in 0.01s ===============================


> ⚠️ If the code inside `with pytest.raises(...)` does **not** raise, the test fails with `DID NOT RAISE`. Keep only the line that should raise inside the block.

### 5.6 Custom failure messages

Add a message after a comma to explain a failure in business terms. It is shown only when the assert fails.

In [19]:
%%writefile test_messages.py
def test_password_length():
    password = "abc123"
    assert len(password) >= 8, f"Password too short: {len(password)} characters, need 8+"

Writing test_messages.py


In [20]:
!python -m pytest test_messages.py -q

F                                                                        [100%]
=================================== FAILURES ===================================
_____________________________ test_password_length _____________________________



    def test_password_length():
        password = "abc123"
>       assert len(password) >= 8, f"Password too short: {len(password)} characters, need 8+"
E       AssertionError: Password too short: 6 characters, need 8+
E       assert 6 >= 8
E        +  where 6 = len('abc123')

test_messages.py:3: AssertionError
=========================== short test summary info ============================
FAILED test_messages.py::test_password_length - AssertionError: Password too short: 6 characters, need 8+
1 failed in 0.02s


### Assertion cheat sheet

| Goal | Assertion |
|---|---|
| Equal | `assert result == expected` |
| Not equal | `assert result != other` |
| True / False | `assert is_valid` / `assert not is_valid` |
| Is None | `assert value is None` |
| Contains | `assert item in collection` |
| Type | `assert isinstance(obj, MyClass)` |
| Float | `assert result == pytest.approx(0.3)` |
| Raises | `with pytest.raises(ValueError, match="msg"): ...` |
| With message | `assert cond, "explanation"` |

## 6. Arrange – Act – Assert (AAA)

Every good test has **three clear parts**:

| Step | Question it answers | What you do |
|---|---|---|
| **Arrange** | *Given* what situation? | Create objects, input data, set up state |
| **Act** | *When* what happens? | Call the **one** function/method under test |
| **Assert** | *Then* what should be true? | Check the result or the new state |

You may also hear it as **Given – When – Then**. Separating the three parts with blank lines (or comments) makes tests easy to read and debug.

### 6.1 Code under test: a shopping cart

In [21]:
%%writefile cart.py
class ShoppingCart:
    def __init__(self):
        self.items = {}          # name -> (price, quantity)

    def add_item(self, name, price, quantity=1):
        if price < 0:
            raise ValueError("Price cannot be negative")
        if quantity <= 0:
            raise ValueError("Quantity must be at least 1")
        _, current_qty = self.items.get(name, (price, 0))
        self.items[name] = (price, current_qty + quantity)

    def remove_item(self, name):
        if name not in self.items:
            raise KeyError(f"{name} is not in the cart")
        del self.items[name]

    def item_count(self):
        return sum(qty for _, qty in self.items.values())

    def total(self):
        return sum(price * qty for price, qty in self.items.values())

    def apply_discount(self, percent):
        if not 0 <= percent <= 100:
            raise ValueError("Discount must be between 0 and 100")
        return round(self.total() * (1 - percent / 100), 2)

Writing cart.py


### 6.2 A test **without** structure (hard to read)

In [22]:
%%writefile test_cart_messy.py
from cart import ShoppingCart


def test_cart():
    c = ShoppingCart(); c.add_item("pen", 50, 2); assert c.total() == 100
    c.add_item("book", 500); assert c.item_count() == 3; c.remove_item("pen")
    assert c.total() == 500; assert c.apply_discount(10) == 450

Writing test_cart_messy.py


Problems:

- It tests **many behaviours** at once. If it fails, which part broke?
- Arrange, act and assert are tangled together.
- The name `test_cart` says nothing about expected behaviour.

### 6.3 The same checks written with AAA

In [23]:
%%writefile test_cart.py
import pytest
from cart import ShoppingCart


def test_new_cart_is_empty():
    # Arrange
    cart = ShoppingCart()

    # Act
    count = cart.item_count()

    # Assert
    assert count == 0


def test_add_item_increases_total():
    # Arrange
    cart = ShoppingCart()

    # Act
    cart.add_item("pen", price=50, quantity=2)

    # Assert
    assert cart.total() == 100


def test_adding_same_item_twice_adds_quantity():
    # Arrange
    cart = ShoppingCart()
    cart.add_item("pen", price=50, quantity=1)

    # Act
    cart.add_item("pen", price=50, quantity=2)

    # Assert
    assert cart.item_count() == 3


def test_remove_item_removes_it_from_total():
    # Arrange
    cart = ShoppingCart()
    cart.add_item("pen", price=50, quantity=2)
    cart.add_item("book", price=500)

    # Act
    cart.remove_item("pen")

    # Assert
    assert cart.total() == 500
    assert "pen" not in cart.items


def test_ten_percent_discount():
    # Arrange
    cart = ShoppingCart()
    cart.add_item("book", price=500)

    # Act
    discounted = cart.apply_discount(10)

    # Assert
    assert discounted == 450


def test_negative_price_is_rejected():
    # Arrange
    cart = ShoppingCart()

    # Act + Assert (the action itself is expected to raise)
    with pytest.raises(ValueError, match="negative"):
        cart.add_item("pen", price=-5)


def test_removing_missing_item_raises_key_error():
    # Arrange
    cart = ShoppingCart()

    # Act + Assert
    with pytest.raises(KeyError):
        cart.remove_item("laptop")

Writing test_cart.py


In [24]:
!python -m pytest test_cart.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-9.1.1, pluggy-1.6.0 -- /usr/local/bin/python
cachedir: .pytest_cache
rootdir: /tmp/claude-0/-home-claude/46131263-cd34-5556-b5d9-de7c2ec56ddb/scratchpad/nb/pytest_lesson
plugins: platformdirs-4.13.5, anyio-4.15.1
collected 7 items                                                              

test_cart.py::test_new_cart_is_empty PASSED                              [ 14%]
test_cart.py::test_add_item_increases_total PASSED                       [ 28%]
test_cart.py::test_adding_same_item_twice_adds_quantity PASSED           [ 42%]
test_cart.py::test_remove_item_removes_it_from_total PASSED              [ 57%]
test_cart.py::test_ten_percent_discount PASSED                           [ 71%]
test_cart.py::test_negative_price_is_rejected PASSED                     [ 85%]
test_cart.py::test_removing_missing_item_raises_key_error PASSED         [100%]

=======================

### 6.4 AAA rules of thumb

1. **One behaviour per test.** Several asserts are fine if they all check the *same* behaviour (as in `test_remove_item_removes_it_from_total`).
2. **Exactly one Act.** If you need two actions, you probably need two tests.
3. **No logic in tests.** Avoid `if`, loops or calculations in the Assert step. Write the expected value literally (`== 450`, not `== 500 * 0.9`), otherwise a bug can hide in both the code and the test.
4. **Tests are independent.** Each test builds its own data in Arrange; never rely on another test having run first.
5. **Name = behaviour.** `test_<what>_<condition>_<expected>`, e.g. `test_discount_over_100_raises_error`.
6. When the act is expected to raise, **Act and Assert merge** inside `pytest.raises`.

## 7. A complete example: bank account

Let's practise the whole flow on a slightly richer class.

In [25]:
%%writefile bank.py
class InsufficientFunds(Exception):
    pass


class BankAccount:
    def __init__(self, owner, balance=0):
        if balance < 0:
            raise ValueError("Opening balance cannot be negative")
        self.owner = owner
        self.balance = balance
        self.history = []

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self.balance += amount
        self.history.append(("deposit", amount))

    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientFunds(f"Balance {self.balance} is less than {amount}")
        self.balance -= amount
        self.history.append(("withdraw", amount))

    def transfer_to(self, other, amount):
        self.withdraw(amount)
        other.deposit(amount)

Writing bank.py


In [26]:
%%writefile test_bank.py
import pytest
from bank import BankAccount, InsufficientFunds


def test_new_account_has_opening_balance():
    # Arrange / Act
    account = BankAccount("Ali", balance=1000)

    # Assert
    assert account.owner == "Ali"
    assert account.balance == 1000
    assert account.history == []


def test_deposit_increases_balance():
    # Arrange
    account = BankAccount("Ali", balance=100)

    # Act
    account.deposit(50)

    # Assert
    assert account.balance == 150


def test_deposit_is_recorded_in_history():
    # Arrange
    account = BankAccount("Ali")

    # Act
    account.deposit(200)

    # Assert
    assert account.history == [("deposit", 200)]


def test_withdraw_more_than_balance_raises():
    # Arrange
    account = BankAccount("Ali", balance=100)

    # Act + Assert
    with pytest.raises(InsufficientFunds, match="less than 500"):
        account.withdraw(500)


def test_failed_withdraw_leaves_balance_unchanged():
    # Arrange
    account = BankAccount("Ali", balance=100)

    # Act
    with pytest.raises(InsufficientFunds):
        account.withdraw(500)

    # Assert
    assert account.balance == 100


def test_transfer_moves_money_between_accounts():
    # Arrange
    sender = BankAccount("Ali", balance=1000)
    receiver = BankAccount("Sara", balance=0)

    # Act
    sender.transfer_to(receiver, 300)

    # Assert
    assert sender.balance == 700
    assert receiver.balance == 300

Writing test_bank.py


In [27]:
!python -m pytest test_bank.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-9.1.1, pluggy-1.6.0 -- /usr/local/bin/python
cachedir: .pytest_cache
rootdir: /tmp/claude-0/-home-claude/46131263-cd34-5556-b5d9-de7c2ec56ddb/scratchpad/nb/pytest_lesson
plugins: platformdirs-4.13.5, anyio-4.15.1
collected 6 items                                                              

test_bank.py::test_new_account_has_opening_balance PASSED                [ 16%]
test_bank.py::test_deposit_increases_balance PASSED                      [ 33%]
test_bank.py::test_deposit_is_recorded_in_history PASSED                 [ 50%]
test_bank.py::test_withdraw_more_than_balance_raises PASSED              [ 66%]
test_bank.py::test_failed_withdraw_leaves_balance_unchanged PASSED       [ 83%]
test_bank.py::test_transfer_moves_money_between_accounts PASSED          [100%]

============================== 6 passed in 0.02s ===============================


## 8. Normal cases, edge cases and invalid input

Testing only the "happy path" is the most common testing mistake. For every function, think about **three kinds of input**:

| Kind | Question | Expected behaviour | Example for `grade(score)` |
|---|---|---|---|
| **Normal cases** | Does it work for typical, everyday input? | Correct result | `grade(85)` → `"B"` |
| **Edge cases** | Does it work at the limits and boundaries? | Correct result, no crash | `grade(0)`, `grade(100)`, `grade(90)` exactly |
| **Invalid input** | Does it reject bad input clearly? | A specific, helpful exception | `grade(-5)`, `grade(150)`, `grade("A")`, `grade(None)` |

> Bugs love boundaries: `<` vs `<=`, off-by-one errors, empty lists, `None`. Normal cases prove the idea works; edge and invalid cases prove the code is **robust**.

### 8.1 Where to look for edge cases

| Input type | Edge cases to try |
|---|---|
| Numbers | `0`, `1`, `-1`, the min and max allowed, **exactly on** a boundary, **just below / just above** it, very large values, floats like `89.99` |
| Strings | `""`, `" "` (only spaces), one character, leading/trailing spaces, mixed case, unicode (`"Zoë"`, `"علی"`), very long text |
| Lists / dicts | empty, one item, duplicates, already sorted / reverse sorted, very large |
| Anything | `None`, wrong type (`"5"` instead of `5`), `True`/`False` (a `bool` is an `int` in Python!) |

**Boundary value analysis:** for a rule like "90 to 100 is an A", test **89.99, 90, 90.01** and **100, 100.01**: just below, on, and just above each boundary.

### 8.2 Code under test: a grading function

In [28]:
%%writefile grades.py
def grade(score):
    # Convert a score from 0 to 100 into a letter grade.
    #   90-100 -> A, 80-89 -> B, 70-79 -> C, 60-69 -> D, below 60 -> F
    if isinstance(score, bool) or not isinstance(score, (int, float)):
        raise TypeError(f"score must be a number, got {type(score).__name__}")
    if score < 0 or score > 100:
        raise ValueError(f"score must be between 0 and 100, got {score}")

    if score >= 90:
        return "A"
    if score >= 80:
        return "B"
    if score >= 70:
        return "C"
    if score >= 60:
        return "D"
    return "F"

Writing grades.py


### 8.3 Testing all three kinds

We group the tests by kind using classes, so the output reads like a specification. Each test still follows **Arrange–Act–Assert**.

In [29]:
%%writefile test_grades.py
import pytest
from grades import grade


class TestNormalCases:
    # Typical scores in the middle of each band

    def test_high_score_is_a(self):
        assert grade(95) == "A"

    def test_mid_score_is_b(self):
        # Arrange
        score = 85

        # Act
        result = grade(score)

        # Assert
        assert result == "B"

    def test_average_score_is_c(self):
        assert grade(75) == "C"

    def test_low_pass_is_d(self):
        assert grade(65) == "D"

    def test_failing_score_is_f(self):
        assert grade(40) == "F"


class TestEdgeCases:
    # Limits of the valid range and every grade boundary

    def test_minimum_score_is_f(self):
        assert grade(0) == "F"

    def test_maximum_score_is_a(self):
        assert grade(100) == "A"

    def test_exactly_on_a_boundary(self):
        assert grade(90) == "A"

    def test_just_below_a_boundary(self):
        assert grade(89.99) == "B"

    def test_pass_mark_boundary(self):
        assert grade(60) == "D"
        assert grade(59.9) == "F"

    def test_float_scores_are_accepted(self):
        assert grade(72.5) == "C"


class TestInvalidInput:
    # Bad input must raise a clear, specific error

    def test_negative_score_raises_value_error(self):
        with pytest.raises(ValueError, match="between 0 and 100"):
            grade(-1)

    def test_score_above_100_raises_value_error(self):
        with pytest.raises(ValueError):
            grade(100.01)

    def test_string_raises_type_error(self):
        with pytest.raises(TypeError, match="must be a number"):
            grade("85")

    def test_none_raises_type_error(self):
        with pytest.raises(TypeError):
            grade(None)

    def test_bool_is_rejected(self):
        # True == 1 in Python, so without a check grade(True) would return "F"
        with pytest.raises(TypeError):
            grade(True)

Writing test_grades.py


In [30]:
!python -m pytest test_grades.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-9.1.1, pluggy-1.6.0 -- /usr/local/bin/python
cachedir: .pytest_cache
rootdir: /tmp/claude-0/-home-claude/46131263-cd34-5556-b5d9-de7c2ec56ddb/scratchpad/nb/pytest_lesson
plugins: platformdirs-4.13.5, anyio-4.15.1
collected 16 items                                                             

test_grades.py::TestNormalCases::test_high_score_is_a PASSED             [  6%]
test_grades.py::TestNormalCases::test_mid_score_is_b PASSED              [ 12%]
test_grades.py::TestNormalCases::test_average_score_is_c PASSED          [ 18%]
test_grades.py::TestNormalCases::test_low_pass_is_d PASSED               [ 25%]
test_grades.py::TestNormalCases::test_failing_score_is_f PASSED          [ 31%]
test_grades.py::TestEdgeCases::test_minimum_score_is_f PASSED            [ 37%]
test_grades.py::TestEdgeCases::test_maximum_score_is_a PASSED            [ 43%]
test_grades.py::TestEdge

Notice how the verbose output now reads like documentation:
`TestEdgeCases::test_just_below_a_boundary PASSED`. Anyone can see which kinds of input are covered.

### 8.4 Why edge cases matter: catching an off-by-one bug

Here is a version of `grade` with a tiny mistake: `>` instead of `>=` for the A band. The **normal** tests all pass, and only the **edge** test catches it.

In [31]:
%%writefile grades_buggy.py
def grade(score):
    if score > 90:          # BUG: should be >= 90
        return "A"
    if score >= 80:
        return "B"
    if score >= 70:
        return "C"
    if score >= 60:
        return "D"
    return "F"

Writing grades_buggy.py


In [32]:
%%writefile test_grades_buggy.py
from grades_buggy import grade


def test_normal_high_score():      # normal case: passes, hides the bug
    assert grade(95) == "A"


def test_normal_mid_score():       # normal case: passes
    assert grade(85) == "B"


def test_edge_exactly_90():        # edge case: FAILS, reveals the bug
    assert grade(90) == "A"

Writing test_grades_buggy.py


In [33]:
!python -m pytest test_grades_buggy.py -v --tb=short

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-9.1.1, pluggy-1.6.0 -- /usr/local/bin/python
cachedir: .pytest_cache
rootdir: /tmp/claude-0/-home-claude/46131263-cd34-5556-b5d9-de7c2ec56ddb/scratchpad/nb/pytest_lesson
plugins: platformdirs-4.13.5, anyio-4.15.1
collected 3 items                                                              

test_grades_buggy.py::test_normal_high_score PASSED                      [ 33%]
test_grades_buggy.py::test_normal_mid_score PASSED                       [ 66%]
test_grades_buggy.py::test_edge_exactly_90 FAILED                        [100%]

=================================== FAILURES ===================================
_____________________________ test_edge_exactly_90 _____________________________
test_grades_buggy.py:13: in test_edge_exactly_90
    assert grade(90) == "A"
E   AssertionError: assert 'B' == 'A'
E     
E     - A
E     + B
=========================== short test 

A student scoring exactly 90 would have been given a **B**. Only the boundary test found it, which is why "it works for 95 and 85" is not enough.

### 8.5 Many cases, one test: `@pytest.mark.parametrize`

Edge-case testing means checking lots of similar inputs. Instead of copy-pasting tests, `parametrize` runs **one test function** once per row of data. Each row appears as its own test in the report.

In [34]:
%%writefile test_grades_parametrized.py
import pytest
from grades import grade


# Normal and edge cases: (score, expected grade)
@pytest.mark.parametrize("score, expected", [
    (95, "A"),      # normal
    (85, "B"),      # normal
    (0, "F"),       # edge: minimum
    (100, "A"),     # edge: maximum
    (90, "A"),      # edge: on boundary
    (89.99, "B"),   # edge: just below boundary
    (60, "D"),      # edge: pass mark
    (59.9, "F"),    # edge: just below pass mark
])
def test_grade_for_valid_scores(score, expected):
    assert grade(score) == expected


# Invalid input: (bad value, expected exception)
@pytest.mark.parametrize("bad_score, error", [
    (-1, ValueError),
    (101, ValueError),
    ("85", TypeError),
    (None, TypeError),
    ([90], TypeError),
    (True, TypeError),
])
def test_grade_rejects_invalid_input(bad_score, error):
    with pytest.raises(error):
        grade(bad_score)

Writing test_grades_parametrized.py


In [35]:
!python -m pytest test_grades_parametrized.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-9.1.1, pluggy-1.6.0 -- /usr/local/bin/python
cachedir: .pytest_cache
rootdir: /tmp/claude-0/-home-claude/46131263-cd34-5556-b5d9-de7c2ec56ddb/scratchpad/nb/pytest_lesson
plugins: platformdirs-4.13.5, anyio-4.15.1
collected 14 items                                                             

test_grades_parametrized.py::test_grade_for_valid_scores[95-A] PASSED    [  7%]
test_grades_parametrized.py::test_grade_for_valid_scores[85-B] PASSED    [ 14%]
test_grades_parametrized.py::test_grade_for_valid_scores[0-F] PASSED     [ 21%]
test_grades_parametrized.py::test_grade_for_valid_scores[100-A] PASSED   [ 28%]
test_grades_parametrized.py::test_grade_for_valid_scores[90-A] PASSED    [ 35%]
test_grades_parametrized.py::test_grade_for_valid_scores[89.99-B] PASSED [ 42%]
test_grades_parametrized.py::test_grade_for_valid_scores[60-D] PASSED    [ 50%]
test_grades_parametrized

### 8.6 A string example: building initials

The same three-way thinking works for text.

In [36]:
%%writefile names.py
def initials(full_name):
    # "Karamat Ali" -> "KA"
    if not isinstance(full_name, str):
        raise TypeError("full_name must be a string")
    parts = full_name.split()          # split() with no argument ignores extra spaces
    if not parts:
        raise ValueError("full_name cannot be empty")
    return "".join(part[0].upper() for part in parts)

Writing names.py


In [37]:
%%writefile test_names.py
import pytest
from names import initials


# Normal cases
def test_first_and_last_name():
    assert initials("Karamat Ali") == "KA"


def test_three_part_name():
    assert initials("Muhammad Ali Jinnah") == "MAJ"


# Edge cases
def test_single_name():
    assert initials("Madonna") == "M"


def test_lowercase_input_is_uppercased():
    assert initials("sara khan") == "SK"


def test_extra_spaces_are_ignored():
    assert initials("  Sara    Khan  ") == "SK"


def test_unicode_name():
    assert initials("Zoë Ólafsdóttir") == "ZÓ"


# Invalid input
def test_empty_string_raises_value_error():
    with pytest.raises(ValueError, match="empty"):
        initials("")


def test_only_spaces_raises_value_error():
    with pytest.raises(ValueError):
        initials("     ")


def test_none_raises_type_error():
    with pytest.raises(TypeError):
        initials(None)

Writing test_names.py


In [38]:
!python -m pytest test_names.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-9.1.1, pluggy-1.6.0 -- /usr/local/bin/python
cachedir: .pytest_cache
rootdir: /tmp/claude-0/-home-claude/46131263-cd34-5556-b5d9-de7c2ec56ddb/scratchpad/nb/pytest_lesson
plugins: platformdirs-4.13.5, anyio-4.15.1
collected 9 items                                                              

test_names.py::test_first_and_last_name PASSED                           [ 11%]
test_names.py::test_three_part_name PASSED                               [ 22%]
test_names.py::test_single_name PASSED                                   [ 33%]
test_names.py::test_lowercase_input_is_uppercased PASSED                 [ 44%]
test_names.py::test_extra_spaces_are_ignored PASSED                      [ 55%]
test_names.py::test_unicode_name PASSED                                  [ 66%]
test_names.py::test_empty_string_raises_value_error PASSED               [ 77%]
test_names.py::test_only

### 8.7 Checklist before you call a function "tested"

- [ ] At least one **normal** case for each kind of typical input
- [ ] The **minimum** and **maximum** allowed values
- [ ] Values **on, just below and just above** every boundary in the rules
- [ ] **Empty** input (`""`, `[]`, `{}`, `0`) and **one-item** input
- [ ] **Wrong types**, including `None` and `bool`
- [ ] **Out-of-range** values, which raise a **specific** exception (`ValueError`, `TypeError`), not a random crash
- [ ] After a failed call, **state is unchanged** (like `test_failed_withdraw_leaves_balance_unchanged` in section 7)

> **Decide the behaviour first.** Edge cases often reveal unanswered questions ("Is a score of 59.5 a pass?"). Agree on the answer, write it as a test, then make the code match.

## 9. Running tests: useful options

| Command | What it does |
|---|---|
| `pytest` | Run every test in the folder tree |
| `pytest test_cart.py` | Run one file |
| `pytest test_cart.py::test_ten_percent_discount` | Run one test by its node id |
| `pytest -v` / `-vv` | Verbose / extra verbose output |
| `pytest -q` | Quiet, compact output |
| `pytest -k "discount"` | Run tests whose name matches an expression |
| `pytest -k "cart and not remove"` | Combine with `and`, `or`, `not` |
| `pytest -x` | Stop at the first failure |
| `pytest --lf` | Re-run only the tests that failed last time |
| `pytest --tb=short` | Shorter tracebacks |
| `pytest --collect-only` | List tests without running them |

Try a few:

In [39]:
!python -m pytest test_cart.py::test_ten_percent_discount -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-9.1.1, pluggy-1.6.0 -- /usr/local/bin/python
cachedir: .pytest_cache
rootdir: /tmp/claude-0/-home-claude/46131263-cd34-5556-b5d9-de7c2ec56ddb/scratchpad/nb/pytest_lesson
plugins: platformdirs-4.13.5, anyio-4.15.1
collecting ... 

collected 1 item                                                               

test_cart.py::test_ten_percent_discount PASSED                           [100%]

============================== 1 passed in 0.01s ===============================


In [40]:
!python -m pytest test_cart.py test_bank.py -k "discount or transfer" -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-9.1.1, pluggy-1.6.0 -- /usr/local/bin/python
cachedir: .pytest_cache
rootdir: /tmp/claude-0/-home-claude/46131263-cd34-5556-b5d9-de7c2ec56ddb/scratchpad/nb/pytest_lesson
plugins: platformdirs-4.13.5, anyio-4.15.1
collected 13 items / 11 deselected / 2 selected                                

test_cart.py::test_ten_percent_discount PASSED                           [ 50%]
test_bank.py::test_transfer_moves_money_between_accounts PASSED          [100%]

======================= 2 passed, 11 deselected in 0.01s =======================


In [41]:
# Run everything we've written; --tb=line keeps each failure to one line
!python -m pytest -q --tb=line

.........................F.......F.....

................F..............F. [ 90%]
........                                                                 [100%]
=================================== FAILURES ===================================
E   AssertionError: assert {'name': 'Ali...ty': 'Lahore'} == {'name': 'Ali...ty': 'Lahore'}
      
      Omitting 3 identical items, use -vv to show
      Differing items:
      {'role': 'admin'} != {'role': 'editor'}
      Use -v to get more diff
/tmp/claude-0/-home-claude/46131263-cd34-5556-b5d9-de7c2ec56ddb/scratchpad/nb/pytest_lesson/test_diff_demo.py:7: AssertionError: assert {'name': 'Ali...ty': 'Lahore'} == {'name': 'Ali...ty': 'Lahore'}
E   assert 0.0 == 180
     +  where 0.0 = apply_discount(200, 10)
/tmp/claude-0/-home-claude/46131263-cd34-5556-b5d9-de7c2ec56ddb/scratchpad/nb/pytest_lesson/test_failure_demo.py:6: assert 0.0 == 180
E   AssertionError: assert 'B' == 'A'
      
      - A
      + B
/tmp/claude-0/-home-claude/46131263-cd34-5556-b5d9-de7c2ec56ddb/scratchpad/nb/pytest

The last run shows the four **intentional** failures: the demos in sections 5.1, 5.3 and 5.6, and the off-by-one bug in 8.4. Everything else passes.

## 10. Common mistakes

| Mistake | Why it's a problem | Fix |
|---|---|---|
| File named `calculator_tests.py` | Not discovered (needs `test_` prefix or `_test.py` suffix) | Rename to `test_calculator.py` |
| `def check_add():` | Not collected | Name it `def test_add():` |
| `assert (a == b, "msg")` | A non-empty tuple is always true: **never fails** | `assert a == b, "msg"` |
| `assert 0.1 + 0.2 == 0.3` | Float rounding | `pytest.approx(0.3)` |
| Many unrelated checks in one test | Hard to see what broke | One behaviour per test (AAA) |
| Tests depend on each other's order | Random failures | Arrange fresh data in every test |
| Computing the expected value with the same formula as the code | The bug is copied into the test | Write expected values literally |

## 11. Exercises

Write the tests yourself first, then compare with the solutions in section 12.

**Code under test** (run this cell first):

In [42]:
%%writefile text_utils.py
def word_count(text):
    return len(text.split())


def is_palindrome(word):
    if not isinstance(word, str):
        raise TypeError("word must be a string")
    cleaned = word.lower().replace(" ", "")
    return cleaned == cleaned[::-1]


def average(numbers):
    if not numbers:
        raise ValueError("Cannot average an empty list")
    return sum(numbers) / len(numbers)

Writing text_utils.py


**Tasks.** Use the AAA pattern in every test.

1. `word_count("hello world from pytest")` returns `4`.
2. `word_count("")` returns `0`.
3. `is_palindrome("Racecar")` is `True` (case-insensitive).
4. `is_palindrome("never odd or even")` is `True` (spaces ignored).
5. `is_palindrome("python")` is `False`.
6. `is_palindrome(123)` raises `TypeError`.
7. `average([1, 2, 3, 4])` equals `2.5`.
8. `average([0.1, 0.2])` is approximately `0.15`.
9. `average([])` raises `ValueError` with a message containing `"empty"`.

**Edge cases and invalid input**

10. `word_count("   lots   of   spaces   ")` returns `3` (edge: extra whitespace).
11. `is_palindrome("a")` and `is_palindrome("")` are both `True` (edge: one character / empty).
12. `average([7])` returns `7` (edge: a single number) and `average([-2, 2])` returns `0` (edge: negatives).
13. `average(None)` raises `ValueError` (invalid input: falsy value) and `is_palindrome(["a"])` raises `TypeError`.
14. **Bonus:** rewrite tasks 3–5 as one test with `@pytest.mark.parametrize`.

Write your tests in the cell below, then run it.

In [43]:
%%writefile test_text_utils_mine.py
import pytest
from text_utils import word_count, is_palindrome, average


# Your tests here, e.g.
def test_word_count_counts_words():
    # Arrange

    # Act

    # Assert
    pass

Writing test_text_utils_mine.py


In [44]:
!python -m pytest test_text_utils_mine.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-9.1.1, pluggy-1.6.0 -- /usr/local/bin/python
cachedir: .pytest_cache
rootdir: /tmp/claude-0/-home-claude/46131263-cd34-5556-b5d9-de7c2ec56ddb/scratchpad/nb/pytest_lesson
plugins: platformdirs-4.13.5, anyio-4.15.1
collected 1 item                                                               

test_text_utils_mine.py::test_word_count_counts_words PASSED             [100%]

============================== 1 passed in 0.01s ===============================


## 12. Solutions

In [45]:
%%writefile test_text_utils.py
import pytest
from text_utils import word_count, is_palindrome, average


def test_word_count_counts_words():
    # Arrange
    text = "hello world from pytest"

    # Act
    result = word_count(text)

    # Assert
    assert result == 4


def test_word_count_of_empty_string_is_zero():
    assert word_count("") == 0


def test_palindrome_ignores_case():
    assert is_palindrome("Racecar") is True


def test_palindrome_ignores_spaces():
    assert is_palindrome("never odd or even") is True


def test_non_palindrome_returns_false():
    assert is_palindrome("python") is False


def test_palindrome_rejects_non_strings():
    with pytest.raises(TypeError):
        is_palindrome(123)


def test_average_of_integers():
    # Arrange
    numbers = [1, 2, 3, 4]

    # Act
    result = average(numbers)

    # Assert
    assert result == 2.5


def test_average_of_floats_is_approximate():
    assert average([0.1, 0.2]) == pytest.approx(0.15)


def test_average_of_empty_list_raises():
    with pytest.raises(ValueError, match="empty"):
        average([])


# ---- Edge cases ----
def test_word_count_ignores_extra_spaces():
    assert word_count("   lots   of   spaces   ") == 3


def test_single_character_is_palindrome():
    assert is_palindrome("a") is True


def test_empty_string_is_palindrome():
    assert is_palindrome("") is True


def test_average_of_single_number():
    assert average([7]) == 7


def test_average_with_negative_numbers():
    assert average([-2, 2]) == 0


# ---- Invalid input ----
def test_average_of_none_raises():
    with pytest.raises(ValueError):
        average(None)


def test_palindrome_rejects_list():
    with pytest.raises(TypeError):
        is_palindrome(["a"])


# ---- Bonus: parametrize ----
@pytest.mark.parametrize("word, expected", [
    ("Racecar", True),
    ("never odd or even", True),
    ("python", False),
])
def test_is_palindrome_cases(word, expected):
    assert is_palindrome(word) is expected

Writing test_text_utils.py


In [46]:
!python -m pytest test_text_utils.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-9.1.1, pluggy-1.6.0 -- /usr/local/bin/python
cachedir: .pytest_cache
rootdir: /tmp/claude-0/-home-claude/46131263-cd34-5556-b5d9-de7c2ec56ddb/scratchpad/nb/pytest_lesson
plugins: platformdirs-4.13.5, anyio-4.15.1
collected 19 items                                                             

test_text_utils.py::test_word_count_counts_words PASSED                  [  5%]
test_text_utils.py::test_word_count_of_empty_string_is_zero PASSED       [ 10%]
test_text_utils.py::test_palindrome_ignores_case PASSED                  [ 15%]
test_text_utils.py::test_palindrome_ignores_spaces PASSED                [ 21%]
test_text_utils.py::test_non_palindrome_returns_false PASSED             [ 26%]
test_text_utils.py::test_palindrome_rejects_non_strings PASSED           [ 31%]
test_text_utils.py::test_average_of_integers PASSED                      [ 36%]
test_text_utils.py::test

> In very short tests, Arrange, Act and Assert can sit on one line (`assert word_count("") == 0`). The structure is still there; comments are optional once the habit is in place.

## 13. Summary

- A **test function** is a function named `test_...` in a file named `test_*.py`; pytest finds and runs it automatically.
- Use plain **`assert`**; pytest explains failures with the real values and detailed diffs.
- Use **`pytest.approx`** for floats and **`pytest.raises`** for expected exceptions.
- Structure every test as **Arrange → Act → Assert**: one behaviour, one action, clear checks.
- Name tests after the behaviour they check, keep them independent, and write expected values literally.
- Cover **normal cases** (typical input), **edge cases** (boundaries, empty, one item, min/max) and **invalid input** (wrong type, out of range, `None`), which should raise specific exceptions.
- Use **`@pytest.mark.parametrize`** to run many inputs through one test.

**Next topics:** fixtures (`@pytest.fixture`) to share Arrange code, mocking with `unittest.mock`, and measuring coverage with `pytest-cov`.

In [ ]:
# Optional clean-up: go back to the original folder
os.chdir("..")
print("Back in:", os.getcwd())